# Explainability and Trust
### Part 2 — Explaining a model you can check

**Author**: Jesper Dramsch (ECMWF)

*This notebook was last tested and operational on 01/10/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was adapted from the author's contribution for the DestinE [2026 Machine Learning for Earth System Modelling Course (Applications and Future Directions)](https://learning.ecmwf.int/enrol/index.php?id=100), where it accompanies the Part 2 video of Module 4, *Explainability & Trust*. It is itself adapted from the [xai-weather-workshop](https://github.com/JesperDramsch/xai-weather-workshop) (CLIMADEMIC Summer School, July 2026, MIT licence), which in turn rebuilt and extended the [ml.recipes](https://github.com/JesperDramsch/ml-for-science-reproducibility-tutorial) interpretability tutorial by Jesper Dramsch and Valerio Maggio (MIT licence).
<!-- ::: -->


<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
You may try to run/access this notebook on the free online platforms linked below. Please note they are not officially supported by or linked with ECMWF.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/Classic_xai.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/Classic_xai.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/Classic_xai.ipynb)

The notebook is self-contained and runs on a CPU. The Palmer Penguins dataset is loaded through the `palmerpenguins` package, and every model is trained in the notebook itself. A version with worked solutions to every exercise is in [`Classic_xai_solved.ipynb`](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/Classic_xai_solved.ipynb).
<!-- 
::: -->


## Introduction

This notebook is a guided, hands-on tour of four classic explainable-AI (XAI) methods, **tree importance**, **permutation importance**, **partial dependence / ICE**, and Shapley analysis (**SHAP**). These methods essentially attempt to explain how important an input feature or a model aspect is to the prediction that an AI model makes. For each method, we will treat three topics:

1. **What question does it answer?**
2. **Worked example**
3. **Where does it mislead you?** Every XAI method has blind spots, and knowing them matters as much as knowing the method.

We will introduce the methods in the context of the [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) dataset: this is a small classification problem: predict penguin species (Adelie, Chinstrap or Gentoo) from bill and flipper measurements, and sex. That lets us focus on the explanation methods rather than the modelling problem.

The notebook then runs three experiments as exercises for you to carry out: add a column of pure noise (Exercises 1 and 2), shuffle the labels and retrain (Exercise 5), drop a feature and watch where its contribution to the output goes (Exercise 6). Each of these experiments works here because the problem we are working on is simple enough that you will know what each experiment is supposed to do. None of the experiments runs on AIFS, which is what Part 3 (`AIFS_sensitivities.ipynb`) is about.

---
## Prepare your environment

This notebook uses a small set of packages:
- `pandas` and `numpy` for data handling
- `matplotlib` and `seaborn` for plotting
- `scikit-learn` for the models, permutation importance and partial dependence
- `shap` for SHAP values
- `palmerpenguins` for the dataset

Run the cell below to install them. This may take a minute.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/c3/m4/requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import (
    PartialDependenceDisplay,
    permutation_importance,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

# Colorblind-safe qualitative palette for every categorical plot below.
sns.set_theme(style="whitegrid", palette="colorblind")
RNG = np.random.RandomState(42)

---
## Load and prepare the data

We load Palmer Penguins from the `palmerpenguins` package, with `seaborn`'s
bundled copy as a fallback if the package or its data file is unreachable
(e.g. a restrictive network). Both sources use the same column names.

We do the minimal cleaning the original analysis relied on: drop rows with
any missing measurement (a handful of penguins have unrecorded sex or
measurements) and keep the four features used throughout this notebook: three
numeric bill/flipper measurements, and `sex` as the one categorical feature.

Finally, we split the dataset into a training and a held-out testing set.

In [ ]:
try:
    from palmerpenguins import load_penguins

    penguins = load_penguins()
except Exception:
    penguins = sns.load_dataset("penguins")

penguins = penguins.dropna().reset_index(drop=True)

num_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
cat_features = ["sex"]
features = num_features + cat_features
target = "species"

X = penguins[features]
y = penguins[target]

print(f"{len(penguins)} penguins after dropping missing values")
penguins.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, train_size=0.7, random_state=42
)
print(f"train: {len(X_train)}, test: {len(X_test)}")

## Train the two models we'll explain

We train two classifiers on identical data through an identical
preprocessing pipeline (scale the numeric features, encode `sex` as a single 0/1 column):
a **Random Forest** and a **Support Vector Classifier**. The random forest is inherently built on tree importance, and it will therefore be our first and simplest method of explaining which inputs matter to obtaining its predictions. The support vector classifier does not have such a built-in metric; its feature importance can *only* be explained with model-agnostic methods like permutation importance and SHAP. We will therefore compare both classifiers.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features),
    ]
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)
rf_model.fit(X_train, y_train)

svc_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(random_state=42)),
    ]
)
svc_model.fit(X_train, y_train)

print(f"Random Forest test accuracy: {rf_model.score(X_test, y_test):.3f}")
print(f"SVM test accuracy:           {svc_model.score(X_test, y_test):.3f}")

---
## 1. Tree (impurity) importance

**The question it answers:** *"Which features did this specific tree-based model lean on internally to reduce prediction error?"* It is read directly off the fitted model, so you need no extra computation. Let us extract it below.

In [ ]:
feature_names_out = rf_model.named_steps["preprocessor"].get_feature_names_out()
feature_names_out = [n.split("__", 1)[1] for n in feature_names_out]  # drop num__/cat__ prefix
importances = rf_model.named_steps["classifier"].feature_importances_

tree_importance = pd.Series(importances, index=feature_names_out).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
tree_importance.plot.bar(ax=ax, color=sns.color_palette("colorblind"))
ax.set_ylabel("Impurity-based importance")
ax.set_title("Tree (impurity) importance (Random Forest)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

The plot would lead you to think that `bill length` and `flipper_length` are the most important two features that explain why the random forest classifier assigns penguins to a species. But this misleads you in several ways:

- **Cardinality bias.** A tree picks the split that reduces impurity most,
  and it can only choose among the candidate split points a feature offers.
  A continuous column like `bill_length` offers hundreds; a binary column like `sex_male` offers
  one. So a continuous feature gets many more chances to look useful by
  accident, and impurity importance rewards it for that. We revisit this in Exercise 1.
- **One-hot dilution**. A categorical feature can have `n_categories` possible values (e.g. a penguin's `sex` can be `male` or `female`). A common way to encode this for classifiers is with *one-hot encoding*, which turns each categorical feature with `n_categories` values into `n_categories` features with binary values. Our dataset has one such one-hot encoded feature. The column `sex`, which originally had labels `male` or `female`, has been turned into two columns, `sex_male` and `sex_female`, with `1` and `0`. However, in the tree impurity importance, the importance of the single feature `sex` is now spread across the two columns, so each column's bar looks smaller than the feature 'deserves'. We sidestep this here by encoding `sex` as a single 0/1 column (`OneHotEncoder(drop="if_binary")` in the cell where the model is trained), which is also why the bar chart above shows one `sex_male` bar rather than `sex_female` and `sex_male`.
- **Correlated features.** A tree only needs to split on *one* of two correlated features to capture their shared signal, so these features will share impurity importance that really only belongs to a single signal.
- It only reflects importance **on the training data the trees were created on**. It says nothing about generalisation, because it's not evaluated again on the test set. So a feature that happens to separate a few noisy training examples can rank highly here while being useless on new data.

### Exercise 1: does impurity importance get fooled by pure noise?

**Task:** add a random-noise feature to the training data (a column of
`np.random.normal` values, completely unrelated to species), retrain the
Random Forest, and re-run tree importance.

*Hint:* `rng = np.random.RandomState(42)` and
`rng.normal(size=len(X_train))`, then `rng.normal(size=len(X_test))`, give you
the same noise column the lecture figure was made from. You'll
need to rebuild the `ColumnTransformer`/`Pipeline` with `noise` added to
`num_features`, the same way `rf_model` was built above.

In [ ]:
# TODO(exercise 1): add a "noise" column to X_train/X_test, retrain a Random
# Forest pipeline on the extended feature set, and plot its tree importance.
# Success criteria: the noise feature has nonzero importance.

# Starter scaffold (already runs as-is) -- extend it below:
X_train_noisy = X_train.copy()
X_test_noisy = X_test.copy()
rng = np.random.RandomState(42)  # fresh generator, so this matches the lecture figure whatever you ran before
X_train_noisy["noise"] = rng.normal(size=len(X_train_noisy))
X_test_noisy["noise"] = rng.normal(size=len(X_test_noisy))

# TODO: build num_features_noisy = num_features + ["noise"], a new
# ColumnTransformer/Pipeline using it, fit on X_train_noisy/y_train, and
# plot tree importance the same way as the cell above.

If you've done it right, the noise feature appears in the bar chart with a *nonzero* importance, despite being pure random noise. Can you explain why, in a sentence, before reading Section 2?

---
## 2. Permutation importance

A slightly more sophisticated and universally applicable method is *permutation importance*.

**The question it answers:** *"If I scramble this one feature's values, how much worse does the model perform?"* Unlike
tree importance, this works for any fitted model. It only needs a model's input, to run it (`.predict`) and to evaluate it (`.score`), not access to the model's internals. That's why it's called *model-agnostic*.

To calculate permutation importance, you therefore scramble a model's input features one at a time, evaluate the model (here first on the training data) and look at how much worse the evaluated result is. It is common to repeat the random scrambling `n_repeats` and look at the *spread* of results, to get an additional sense of uncertainty in the permutation importance (we could not get this from impurity importance).

In [ ]:
rf_perm = permutation_importance(rf_model, X_train, y_train, n_repeats=10, random_state=42)

perm_importance = pd.Series(rf_perm.importances_mean, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
perm_importance.plot.bar(ax=ax, color=sns.color_palette("colorblind"))
ax.set_ylabel("Mean decrease in accuracy")
ax.set_title("Permutation importance (Random Forest)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

Notice `sex` is now a single bar. Permutation importance can operate directly on categorical features, so it doesn't suffer from the one-hot encoding problem in Section 1. 

Now let us evaluate the permutation importance on the SVM, where we could not calculate impurity importance. We also switch to a boxplot here to show all 10 repeats, not just their mean.

In [ ]:
svc_perm = permutation_importance(svc_model, X_train, y_train, n_repeats=10, random_state=42)

fig, ax = plt.subplots(figsize=(7, 5))
pd.DataFrame(svc_perm.importances.T, columns=features).plot.box(ax=ax, color=dict(boxes="C0", medians="C1"))
ax.set_ylabel("Decrease in accuracy per repeat")
ax.set_title("Permutation importance from 10 repeats (SVM)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Performance on training and test datasets for both models

A model that overfits will often show *high permutation importance on the training set that doesn't hold up on the test set*, because the model learned to rely on patterns specific to the training data, e.g. wiggles in only one of the inputs which mapped particularly well onto the output of a training case. Comparing training-set and test-set permutation importance side by side is therefore a fast, visual overfitting check. If you additionally compare two different model families, you can additionally get a sense of whether a feature's importance to a prediction is a property of the *data* or an artifact of one particular *model*.

In [ ]:
rf_perm_train = permutation_importance(rf_model, X_train, y_train, n_repeats=10, random_state=42)
rf_perm_test = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)
svc_perm_train = permutation_importance(svc_model, X_train, y_train, n_repeats=10, random_state=42)
svc_perm_test = permutation_importance(svc_model, X_test, y_test, n_repeats=10, random_state=42)


def _to_long_df(result, group_label):
    df = pd.DataFrame(result.importances.T, columns=features)
    df["Group"] = group_label
    return df


combined = pd.concat(
    [
        _to_long_df(rf_perm_train, "Random Forest (train)"),
        _to_long_df(rf_perm_test, "Random Forest (test)"),
        _to_long_df(svc_perm_train, "SVM (train)"),
        _to_long_df(svc_perm_test, "SVM (test)"),
    ]
)
melted = pd.melt(combined, id_vars=["Group"], var_name="Feature", value_name="Permutation importance")

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(x="Feature", y="Permutation importance", hue="Group", data=melted, palette="colorblind", ax=ax)
ax.set_title("Permutation importance: Random Forest vs. SVM, train vs. test")
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

This plot is quite insightful, because it tells you that while the bill length is a relatively stably important input feature across model and set, bill depth is not: It varies a lot between the random forest (less important) and the SVM (more important). And flipper length and sex are consistently more important on the test dataset than on the training dataset, i.e. they are clearly not features the model has overfit on. However, permutation importance too comes with pitfalls.

**Where permutation importance misleads you.**

- **Correlated features.** Just like in tree importance: if two features are correlated, scrambling one alone barely hurts performance, since the model still has the other one to lean on. So both can look artificially *unimportant* even though the pair together matters a lot.
- **It's expensive.** You re-score the model on the entire dataset for every feature `n_repeats` times. That can become an issue with large datasets or slow models.
- **It answers "how much does performance drop", not "why".** Two features can have identical permutation importance for entirely different reasons.

### Exercise 2: does permutation importance resist the noise trap?

**Task:** repeat the Exercise 1 noise-feature experiment (you'll have
`X_train_noisy`/`X_test_noisy` already if you did Exercise 1, or rebuild
them), but this time run **permutation importance** instead of tree
importance, on the Random Forest. Do it on the held-out split as well as the
training split; the lecture figure uses the held-out one.

Once you've implemented it, compare the noise feature's permutation importance to its tree importance from Exercise 1. Are they still both "nonzero" in a way that could fool you? Write one sentence explaining the disagreement (or agreement) before moving on.

In [ ]:
# TODO(exercise 2): rebuild a Random Forest pipeline on the noisy feature set
# (reuse X_train_noisy/X_test_noisy from Exercise 1) and run
# permutation_importance on it. Compare the "noise" feature's value here to
# its tree-importance value in Exercise 1.

# Starter scaffold (already runs as-is) -- extend it below:
num_features_noisy = num_features + ["noise"]
features_noisy = num_features_noisy + cat_features

# TODO: build the ColumnTransformer/Pipeline for num_features_noisy, fit it
# on X_train_noisy/y_train, then call permutation_importance(...) on it and
# plot the result the same way as the RF permutation importance cell above.
# Careful with the labels: permutation_importance returns one value per column
# of the DataFrame you pass in, in that DataFrame's column order, so index the
# result with X_train_noisy.columns, not with features_noisy.

---
## 3. Partial dependence plot (PDP) and individual conditional expectation (ICE)

**The question it answers:** *"On average, how does the model's predicted
probability change as I vary one feature, holding everything else fixed?"*
That's called a **partial dependence plot (PDP)**. It shows an *averaged*
response curve, e.g. the average fraction of penguins that are Adelie as a function of bill length. **ICE curves** show the same thing, but *per individual data point*, i.e. before averaging. This allows you to see whether the average curve is representative of whether Adelie fraction varies with bill length the same way at e.g. all flipper lengths, or if it's an average of very different sensitivities at different flipper length (and other features).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, species in zip(axs, rf_model.classes_):
    PartialDependenceDisplay.from_estimator(
        rf_model, X_train, num_features, target=species, ax=ax
    )
    ax.set_title(f"PDP: P({species})")
plt.tight_layout()
plt.show()

Now let's add ICE curves for one feature/class combination, to see what the
averaging in the plots above was hiding.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
PartialDependenceDisplay.from_estimator(
    rf_model,
    X_train,
    ["flipper_length_mm"],
    target="Gentoo",
    kind="both",
    ax=ax,
    ice_lines_kw={"color": "C0", "alpha": 0.15},
    pd_line_kw={"color": "C1", "linewidth": 3},
)
ax.set_title("PDP (thick) + ICE curves (thin) -- P(Gentoo) vs. flipper length")
plt.tight_layout()
plt.show()

The PDP is a very simple tool that immediately gives intuition about what our classifier does. For example, we saw it relies strongly on bill length, and the PDP suggests why: It seems very helpful to the model in distinguishing between Adelie (which on average have small bill length) and Chinstrap (which have large ones). It also depended strongly on flipper length, and that seems to be useful in distinguishing Gentoos (which it suggests there are many of at large flipper lengths) from the rest (of which there are few at large flipper length). The ICE curves also confirm this is relatively general across bill lengths, but also points at:

**Where it misleads you.**

- **Averaging hides interactions.** The thick PDP line in the plot above is
  the *average* of all the thin ICE lines. But the average masks a high sensitivity around 205 mm for one group of bill lengths, and a weak response for another group. On average, the line doesn't seem representative of any of the two actual groups.
- **Extrapolation into empty feature space.** To create a PDP for a particular feature, you have to evaluate the model at every value of that feature several times: E.g. while holding bill length fixed, you evaluate the model at all other possible values of flipper length in the dataset, and you do that at every bill length.  But that means you may test combinations of bill length and flipper length that do not occur on any real penguin (e.g. a very long bill paired with a very short flipper). The model will still make a prediction there, but it's extrapolating into a region with no training data to support it. The PDP curve doesn't warn you of this.
- Both PDP and ICE only vary **one (or two) features at a time**. So they can't tell you about higher-order interactions directly.

### Exercise 3: find a penguin whose ICE curve disagrees with the average

**Task:** add ICE curves for `bill_depth_mm` (pick whichever species class
you like as the `target`) and find an individual penguin whose ICE curve
diverges noticeably from the average PDP line.

Then identify (by row index or by printing its features) at
least one penguin whose individual curve visibly disagrees with the thick
average line, and write a one-sentence hypothesis for *why* that penguin
behaves differently (hint: look at its *other* feature values).

*Hint:* `PartialDependenceDisplay.from_estimator(..., kind="both")` returns a
`display` object; its `.pd_results[0]["individual"]` array contains every
sampled row's ICE curve values for every class (index it with `display.target_idx`
to get the class you plotted), and `.pd_results[0]["grid_values"][0]` holds
the corresponding x-axis grid, which is useful if you want to find the
most-diverging curve using code.

In [ ]:
# TODO(exercise 3): plot PDP+ICE (kind="both") for "bill_depth_mm" the same
# way as the flipper_length_mm example above, then find an individual
# penguin (a single ICE line) whose curve diverges from the average PDP
# line. Print that penguin's row from X_train to see what makes it unusual.

# Starter scaffold (already runs as-is) -- extend it below:
fig, ax = plt.subplots(figsize=(6, 5))
PartialDependenceDisplay.from_estimator(
    rf_model,
    X_train,
    ["bill_depth_mm"],
    target="Chinstrap",
    kind="average",  # TODO: change to "both" once you're ready to see ICE curves
    ax=ax,
)
plt.tight_layout()
plt.show()

---
## 4. SHAP

The [Shapley value](https://en.wikipedia.org/wiki/Shapley_value) is a method of trying to fairly attribute contributions from different inputs to a result, which originates in game theory. It has become very popular in explainable AI, and is often abbreviated and capitalised to SHAP.

**How Shapley values work.** To make one prediction, such as the model's P(Gentoo) for penguin #99, you can ask: "which features determined this P(Gentoo), relative to the average P(Gentoo)?" For example, to assess the importance of flipper length, SHAP calculates the model's P(Gentoo) when it is given *this penguin's* flipper length, against its P(Gentoo) when flipper length is filled in from other penguins. However, how much having the right flipper length changes the prediction depends on which other measurements of penguin #99 the model already knows: bill length, bill depth, nothing... The Shapley value is therefore an average over all orders in which the features could be revealed to the model. The result, −0.20 for flipper length (i.e. it is short for a Gentoo), tells you how far that feature pushed this penguin's P(Gentoo) below the average P(Gentoo) over all penguins (0.36). The values for all features (0.09 for bill length, 0.29 for bill depth) add up exactly to the gap between this penguin's P(Gentoo) issued by the model (0.54) and that average: (0.09 + 0.29 − 0.20 = 0.54 − 0.36 = 0.18), in this case a positive deviation (more likely than average to be a Gentoo).

**Mean |SHAP|** turns these per-penguin values into one ranking for the whole model. For each feature, it averages the size of its Shapley values over all test penguins and all three species, ignoring sign, so that pushes towards and away from a species don't cancel out. A large mean |SHAP|, as for bill length here, means the feature moves the predicted probabilities a lot. So:

**The question SHAP answers:** First, *"for one prediction, how much did each feature push the output away from the average prediction across the dataset?"* and, aggregated across many predictions, *"which features matter most overall, and in which direction?"*.

To see SHAP in action, we train a dedicated Random Forest on the three numeric features only. This avoids treating categorical values, which are more intricate for SHAP. We give the model a distinct name, `shap_rf`, to avoid confusing it with `rf_model` above (which includes `sex` and is used everywhere else in this notebook).

In [ ]:
shap_rf = RandomForestClassifier(random_state=42)
shap_rf.fit(X_train[num_features], y_train)
print(f"shap_rf test accuracy: {shap_rf.score(X_test[num_features], y_test):.3f}")
print(f"classes: {list(shap_rf.classes_)}")

explainer = shap.Explainer(shap_rf)
shap_values = explainer(X_test[num_features])
print(f"shap_values.values.shape: {shap_values.values.shape}  (n_samples, n_features, n_classes)")

### Shapley values: why did the model predict *this* species for *this* penguin?

To visualise how Shapley values lead to the predicted class of penguin #99, you can use a waterfall plot: Starting from the average predicted probability (`E[f(X)]`=0.36), each feature pushes P(Gentoo) up (red) or down (blue) until we reach the model's actual output for this one penguin.

In [ ]:
sample_idx = 99
predicted_class = shap_rf.predict(X_test[num_features].iloc[[sample_idx]])[0]
class_idx = list(shap_rf.classes_).index(predicted_class)

print(f"Penguin #{sample_idx}: true species = {y_test.iloc[sample_idx]}, predicted = {predicted_class}")
shap.plots.waterfall(shap_values[sample_idx, :, class_idx], show=False)
plt.title(f"SHAP waterfall -- P({predicted_class}) for penguin #{sample_idx}")
plt.tight_layout()
plt.show()

So, for penguin #99, the SHAP analysis indicates that the model predicts it is a Gentoo, largely because of its shallow bill depth, and to a smaller extent because of its longer-than-average bill length.

### Mean SHAP: which features matter most overall?

To visualise the relative importance of each feature for *all* penguins, you can use a beeswarm plot. In it, each dot is one 
penguin from the test set, and its position shows that feature's SHAP value (impact on the predicted probability for one class), and its colour shows whether that penguin's *own* feature value was high or low. In this sense, it does a similar job as the PDP/ICE curves: E.g. it tells you that the model is unlikely to predict a Gentoo at small flipper length (consistently low feature value at low SHAP). However, it also tells you that relative to other features, the SHAP values get particularly negative, i.e. it is particularly important.

In [ ]:
class_idx = list(shap_rf.classes_).index("Gentoo")

shap.plots.beeswarm(shap_values[:, :, class_idx], show=False)
plt.title("SHAP summary -- P(Gentoo)")
plt.tight_layout()
plt.show()

**Where it misleads you.**

- **Explaining the model is not explaining the world.** SHAP tells you exactly how *this fitted model* used each feature. It does not say  whether the model itself is a good description of penguin biology. If `sex` correlates with body size in this dataset (it does) and the model uses this, SHAP would report "`sex` mattered". That would correctly describe the model's behaviour, but not tell you why sex and species are correlated in the real world.
- **Background/reference data matters.** SHAP values are relative to a baseline distribution (here, the training data). If you change the reference set the same prediction can get a different attribution breakdown.
- **Exact Shapley values are expensive to compute.** Often, you need approximations (see `shap.Explainer`'s auto-selected
  algorithm).

### Exercise 4: SHAP vs. permutation importance -- do the rankings agree?

**Task:** compute SHAP's global feature ranking (mean absolute SHAP value
across all classes) for the three numeric features, and compare it to
permutation importance's ranking *restricted to the same three numeric
features* (use `perm_importance` from Section 2 -- SHAP here only saw
`num_features`, so restrict the comparison to make it fair).

Produce a small table or pair of bar charts showing
both rankings side by side, and write one sentence on where they agree or
disagree, and why that might be (hint: think about correlated features).

*Hint:* `shap_values.values` has shape `(n_samples, n_features, n_classes)`
-- averaging `np.abs(...)` over samples *and* classes (`axis=(0, 2)`) gives
one global importance number per feature, comparable in spirit to
`perm_importance`.

In [ ]:
# TODO(exercise 4): compute mean absolute SHAP value per feature (average
# over samples and classes), compare its ranking to perm_importance
# (restricted to num_features), and plot both side by side.

# Starter scaffold (already runs as-is) -- extend it below:
shap_importance = pd.Series(
    np.abs(shap_values.values).mean(axis=(0, 2)), index=num_features
).sort_values(ascending=False)
print("SHAP global importance (numeric features):")
print(shap_importance)

# TODO: pull perm_importance.loc[num_features] from Section 2, rank both,
# and compare (a simple pd.DataFrame with both rank columns is enough).

### Exercise 5: does SHAP notice when the model has learned nothing?

This is the experiment from Part 4 of the module, but you may already carry it out now.

**Task:** shuffle `y_train` at random, train a fresh Random Forest on
`X_train[num_features]` with the shuffled labels, and report its test
accuracy next to the majority-class baseline. Then compute the mean absolute
SHAP value per feature exactly as in Exercise 4, and plot it beside
`shap_importance` (the real model's ranking) on **one shared x-axis**.

If you do it right, you should get a model that performs no better than always guessing the most common species (about 40% correct, against a 44% majority-class baseline). Every one of its bars on the bar chart should have SHAP quantities that are smaller than the real model's smallest bar (because the random noise shouldn't change the predictions in a biased way). Then write one sentence on what in the figure actually tells you the model is useless. (Hint: it is not the
bars.)

*Hint:* use `np.random.RandomState(42).permutation(y_train.values)` so your
numbers match the lecture.


In [ ]:
# TODO(exercise 5): train on shuffled labels, explain with SHAP, plot beside the real ranking.

# Starter scaffold (already runs as-is) -- extend it below:
rng_shuffle = np.random.RandomState(42)
y_train_shuffled = pd.Series(rng_shuffle.permutation(y_train.values), index=y_train.index)
majority_baseline = y_test.value_counts(normalize=True).max()
print(f"majority-class baseline on the test set: {majority_baseline:.2f}")

# TODO: fit shap_rf_shuffled = RandomForestClassifier(random_state=42) on
# X_train[num_features] / y_train_shuffled, print its test accuracy, explain
# X_test[num_features] with shap.Explainer, average np.abs(...) over samples
# and classes (axis=(0, 2)), and plot the result next to shap_importance with
# the same xlim on both axes.


### Exercise 6: drop `sex`, refit, and watch where the credit goes

Because the penguin models are cheap to train and have so few input features, we can actually just drop a feature, train again and check what it does to the model's performance. For a weather model like AIFS, you cannot perform tests like these, because it's prohibitively expensive to systematically drop inputs, and then retrain the model every time. But here we will do it because it reveals something new to be aware of.

**Task:** build the same Random Forest pipeline on `num_features` only (no
`sex`), compare its test accuracy with `rf_model`, and compare permutation
importance on the **test** split for the three remaining features before and
after dropping `sex`.

If you did it right, the accuracy should barely move, and the permutation
importance that `sex` had should reappear almost entirely under one of the
remaining features. Write one sentence on which feature absorbed it and what
that says about what `sex` was encoding. (Hint: `sex` is balanced within
every species, so on its own it says nothing about species. Check with
`pd.crosstab(y, X["sex"], normalize="index")`.)


In [ ]:
# TODO(exercise 6): refit without sex, compare accuracy and test-split permutation importance.

# Starter scaffold (already runs as-is) -- extend it below:
print(pd.crosstab(y, X["sex"], normalize="index").round(2))

rf_perm_test_with_sex = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)
perm_with_sex = pd.Series(rf_perm_test_with_sex.importances_mean, index=features)
print("\npermutation importance on the test split, with sex:")
print(perm_with_sex.round(3))

# TODO: build rf_no_sex, a Pipeline with a ColumnTransformer over num_features
# only, fit it on X_train[num_features] / y_train, print its test accuracy next
# to rf_model's, run permutation_importance on X_test[num_features], and put the
# two Series side by side in a DataFrame.


---
## Recap

We have introduced four often-used explainable AI methods and their pitfalls:

| Method | Answers | Model-agnostic? | Main pitfall |
|---|---|---|---|
| Tree importance | What did *this tree model* rely on? | No | Cardinality bias; training data only |
| Permutation importance | How much does scrambling *this feature* hurt performance? | Yes | Correlated features share blame; expensive |
| PDP / ICE | How does the prediction change as *this feature* varies? | Yes | Averaging hides interactions; extrapolation |
| SHAP | How much did *this feature* contribute to *this prediction*? | Depends on explainer | Explains the model, not the world |

As you'll notice, three of these four methods -- permutation importance, PDP/ICE, and SHAP --
are *perturbation-based*: they probe the model with different inputs
(permuted, varied one at a time, or combined) and observe
how the output responds. So they treat the model itself as a black box and don't say anything about how it internally converts the inputs to outputs. Tree importance is the exception: it is read directly off the fitted model's internal impurity-reduction bookkeeping.

Part 3 of this module (`AIFS_sensitivities.ipynb`) applies a fundamentally different,
**gradient-based** approach (backward/adjoint sensitivities) to ECMWF's
AIFS operational weather model. If you've worked with adjoint models or 4D-Var
data assimilation in numerical weather prediction, that method will look
familiar.